In [10]:
import sys
import os
import json
import re
import shutil
import logging

logging.disable(logging.CRITICAL)

# ==============================================================================
TARGET_FILE = "irl_mk_articles_dataset.json"
# ==============================================================================

os.environ["PYTHONIOENCODING"] = "utf-8"
sys.modules["torch"] = None

from loguru import logger
logger.disable("datatrove")

from datatrove.executor import LocalPipelineExecutor
from datatrove.pipeline.readers.base import BaseDiskReader
from datatrove.pipeline.writers import JsonlWriter
from datatrove.data import Document

STATS = {"read": 0, "saved": 0, "dropped": 0}

def macedonian_filter(data, *args):
    for doc in data:
        STATS["read"] += 1
        if not doc.text:
            STATS["dropped"] += 1
            continue
        cyrillic = len(re.findall(r'[абвгдѓежзѕијклљмнњопрстќуфхцчџш]', doc.text.lower()))
        total    = len(re.findall(r'[a-zа-ш]', doc.text.lower()))
        if total > 0 and (cyrillic / total) > 0.5:
            yield doc
        else:
            STATS["dropped"] += 1

def length_filter(data, *args):
    for doc in data:
        if len(doc.text.split()) >= 200:
            STATS["saved"] += 1
            yield doc
        else:
            STATS["dropped"] += 1

class UniversalReader(BaseDiskReader):
    def read_file(self, filepath):
        with self.data_folder.open(filepath, "r", encoding="utf-8") as f:
            if filepath.endswith(".jsonl"):
                for i, line in enumerate(f):
                    try:
                        entry = json.loads(line)
                        text = entry.get("text") or entry.get("content") or ""
                        text = re.sub(r'<[^>]+>', '', text)
                        text = re.sub(r'\s+', ' ', text).strip()
                        if text:
                            yield Document(text=text, id=f"{filepath}_{i}", metadata=entry)
                    except Exception:
                        continue
            else:
                try:
                    data = json.load(f)
                    if isinstance(data, list):
                        for i, entry in enumerate(data):
                            text = entry.get("text") or entry.get("content") or ""
                            text = re.sub(r'<[^>]+>', '', text)
                            text = re.sub(r'\s+', ' ', text).strip()
                            if text:
                                yield Document(text=text, id=f"{filepath}_{i}", metadata=entry)
                except Exception as e:
                    print(f"[WARNING] Грешка при читање: {e}")

def run():
    full_path = os.path.join("raw_data", TARGET_FILE)
    if not os.path.exists(full_path):
        print(f"[ERROR] Фајлот '{TARGET_FILE}' не е во 'raw_data'!")
        return

    dataset_name = os.path.splitext(TARGET_FILE)[0]
    output_dir   = os.path.join("clean_data", dataset_name)

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    print(f">>> Обработка: {TARGET_FILE}")

    pipeline = [
        UniversalReader(data_folder="raw_data", glob_pattern=TARGET_FILE),
        macedonian_filter,
        length_filter,
        JsonlWriter(output_folder=output_dir, compression=None),
    ]

    LocalPipelineExecutor(pipeline=pipeline, tasks=1, logging_dir=None).run()

    print(f"\n{'─'*40}")
    print(f"  Влез:      {STATS['read']}")
    print(f"  Исфрлени:  {STATS['dropped']}")
    print(f"  Зачувани:  {STATS['saved']}")
    print(f"{'─'*40}\n")

if __name__ == "__main__":
    run()

>>> Обработка: irl_mk_articles_dataset.json

────────────────────────────────────────
  Влез:      393
  Исфрлени:  13
  Зачувани:  380
────────────────────────────────────────

